# Proyecto Final Part 2 (AFN convertido a AFD) — Validador de direcciones IPv4
## Contexto y reglas
El proyecto utiliza dos autómatas finitos para analizar
direcciones IPv4. En este apartado nos enfocaremos solomaente en el AFN.
**AFN (L2):** Reconoce 4 octetos de 0-255 y entre cada octeto tiene que haber el "." .
Alfabeto: {0,1,2,3,4,5,6,7,8,9,.}
## 3.3. Definición y simulación del AFN
El AFN reconoce cadenas formadas por cuatro componentes numéricos separados por puntos. Los primeros tres componentes pueden tener entre uno y tres dígitos, mientras que el último debe representar un valor entre 0 y 255, sin ceros iniciales innecesarios.

## Cambios que se haran en el AFN
Nos fijamos que el AFN valida cualquier numero en los primeros tres octetos con que solamente sean 3 digitos maximo. Entonces, cambiaremos la logica a que valide cada octeto de 0 a 255.

In [1]:
# Definicion del AFN
alfabeto = {"0", "1", "2", "3", "4",
            "5", "6", "7", "8", "9", "."}
AFN = {
    "estados": set(),
    "inicial": "A0",
    "alfabeto": alfabeto,
    "aceptacion": set(),
    "delta": {}
}


### Función de transición del AFN
Las transiciones permiten leer entre uno y tres dígitos en cada uno de los primeros tres componentes.
Después del tercer punto, el autómata presenta una ramificación no determinista hacia dos estados diferentes. Una rama reconoce el valor 0 y la otra reconoce los valores del 1 al 255.

In [2]:
# Transiciones del AFN
digitos = "0123456789"
def agregar_octeto(prefijo, es_ultimo=False, siguiente_inicial=None):
    """
    Construye un subautomata que reconoce enteros en [0,255]
    sin ceros a la izquierda (salvo el propio 0)

    Args:
        prefijo (_type_): _description_
        es_ultimo (bool, optional): _description_. Defaults to False.
        siguiente_inicial (_type_, optional): _description_. Defaults to None.
    
    Estados: prefijo+0...prefijo+12.
    - es_ultimo=True -> prefijo+1..12 son de aceptacion (sin salida).
    - es_ultimo=False -> prefijo+1..12 transiton con "." a siguiente_inicial.
    """
    
    for i in range(13):
        AFN["estados"].add(f"{prefijo}{i}")
    
    def t(origen, simbolo, destinos):
        AFN["delta"][(origen,simbolo)] = set(destinos)
    
    #primer digito
    t(f"{prefijo}0", "0", [f"{prefijo}1"])               # 0
    t(f"{prefijo}0", "1", [f"{prefijo}2"])               # 1x-19x
    t(f"{prefijo}0", "2", [f"{prefijo}3"])               # 2x / 20x-25x
    for d in "3456789":
        t(f"{prefijo}0", d, [f"{prefijo}4"])             #3x-9x
    
    #segundo digito
    for d in digitos:
        t(f"{prefijo}2", d, [f"{prefijo}5"])             # 10-19 / 100-199
        t(f"{prefijo}4", d, [f"{prefijo}9"])             # 30-99
        t(f"{prefijo}5", d, [f"{prefijo}10"])            # 100-199 (3.er díg.)
        t(f"{prefijo}6", d, [f"{prefijo}11"])            # 200-249 (3.er díg.)
    
    for d in "01234":
        t(f"{prefijo}3", d,[f"{prefijo}6"])              # 20-24 / 200-249
    t(f"{prefijo}3", "5", [f"{prefijo}7"])               # 25 / 250-255
    for d in "6789":
        t(f"{prefijo}3", d, [f"{prefijo}8"])              # 26-29 (26x-29x se rechazan)
    
    for d in ("012345"):
        t(f"{prefijo}7", d, [f"{prefijo}12"])            # 250-255 
        
    #cierre del octeto
    for i in range(1,13):
        if es_ultimo:
            AFN["aceptacion"].add(f"{prefijo}{i}")
        elif siguiente_inicial is not None:
            t(f"{prefijo}{i}", ".", [siguiente_inicial])

#encadenar los cuatro octetos            
agregar_octeto("A", es_ultimo=False, siguiente_inicial="B0")
agregar_octeto("B", es_ultimo=False, siguiente_inicial="C0")
agregar_octeto("C", es_ultimo=False, siguiente_inicial="D0")
agregar_octeto("D", es_ultimo=True)

print("Estados:", len(AFN["estados"]))
print("Inicial:", AFN["inicial"])
print("Finales:", sorted(AFN["aceptacion"]))
print("Transiciones creadas:", len(AFN["delta"]))

Estados: 52
Inicial: A0
Finales: ['D1', 'D10', 'D11', 'D12', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9']
Transiciones creadas: 300


### Simulador del AFN
El simulador inicia con un conjunto que contiene el estado inicial y procesa la cadena símbolo por símbolo.
Para cada símbolo se consultan las transiciones de todos los estados activos y se reúnen sus destinos en un nuevo conjunto.
Al finalizar la lectura, la cadena será aceptada si al menos uno de los estados activos pertenece al conjunto de aceptación.

In [3]:
#Simulador del AFN
def leeCadena(AFN, cadena, debug=False):
    activos = {AFN["inicial"]}

    for posicion, simbolo in enumerate(cadena, start=1):
        if simbolo not in AFN["alfabeto"]:
            if debug:
                print(f"Símbolo fuera del alfabeto en posición {posicion}: {simbolo!r}")
            return False

        siguiente = set()
        for estado in activos:
            siguiente.update(AFN["delta"].get((estado, simbolo), set()))

        if not siguiente:
            if debug:
                print(
                    f"Sin transición en posición {posicion}: "
                    f"estado(s) {sorted(activos)}, símbolo {simbolo!r}"
                )
            return False

        activos = siguiente

    aceptada = bool(activos & AFN["aceptacion"])
    if debug and not aceptada:
        print(f"Terminó en estado(s) no finales: {sorted(activos)}")

    return aceptada

## Simulador del AFN
En este apartado se realizaran pruebas con diferentes tipos de cadena y de esa manera lograr ver si las acepta o rechaza 

In [4]:
#Pruebas del AFN
pruebas_afn = [
    "192.168.1.100",
    "10.0.0.255",
    "172.16.0.50",
    "224.1.2.0",
    "300.1.2.25",
    "192.168.1.256",
    "10.0.0.999",
    "172.16.1.01",
    "192.168.1",
    "192.168.1.1000",
    "",
    "192.168.A.255"
]

print("RESULTADOS DEL AFN")
for cadena in pruebas_afn:
    print("\nCadena:", repr(cadena))
    if any(s not in AFN["alfabeto"] for s in cadena):
        print("FUERA DEL ALFABETO")
    elif leeCadena(AFN, cadena):
        print("ACEPTADA")
    else:
        print("RECHAZADA")

RESULTADOS DEL AFN

Cadena: '192.168.1.100'
ACEPTADA

Cadena: '10.0.0.255'
ACEPTADA

Cadena: '172.16.0.50'
ACEPTADA

Cadena: '224.1.2.0'
ACEPTADA

Cadena: '300.1.2.25'
RECHAZADA

Cadena: '192.168.1.256'
RECHAZADA

Cadena: '10.0.0.999'
RECHAZADA

Cadena: '172.16.1.01'
RECHAZADA

Cadena: '192.168.1'
RECHAZADA

Cadena: '192.168.1.1000'
RECHAZADA

Cadena: ''
RECHAZADA

Cadena: '192.168.A.255'
FUERA DEL ALFABETO
